# Backtest reproducible de señales técnicas

Este notebook compara una estrategia cuantitativa sencilla con una cartera equiponderada. La muestra incluida es **sintética** y sirve para comprobar el pipeline sin Internet; no representa resultados reales de inversión.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import run_backtest
from src.data import load_prices
from src.reporting import save_outputs
from src.signals import build_strategy_weights

## Protocolo

Los factores usan únicamente datos pasados y presentes. Las posiciones se retrasan una sesión, el 30 % final se reserva para evaluación y se descuentan 10 puntos básicos por unidad de rotación.

In [ ]:
prices = load_prices(ROOT / 'data' / 'sample_prices.csv')
print(f'{len(prices):,} sesiones entre {prices.index.min().date()} y {prices.index.max().date()}')
print(prices.tail().round(2).to_string())

In [ ]:
scores, weights = build_strategy_weights(prices, top_n=2)
daily, metrics = run_backtest(prices, weights, cost_bps=10, test_size=0.30)
print(metrics.round(4).to_string())

In [ ]:
latest_scores = scores.dropna(how='all').iloc[-1]
print('Score final (heurístico):')
print(latest_scores.sort_values(ascending=False).round(4).to_string())

ax = daily[['strategy_equity', 'benchmark_equity']].plot(
    figsize=(10, 5), title='Crecimiento fuera de muestra de 1 €'
)
ax.set_ylabel('Capital acumulado')
ax.grid(alpha=0.25)

save_outputs(daily, metrics, latest_scores, ROOT / 'reports' / 'generated')

## Interpretación responsable

Una diferencia favorable en una única partición no demuestra capacidad predictiva. Antes de usar una estrategia con datos reales serían necesarias validaciones walk-forward, análisis de sensibilidad, costes más completos e intervalos de incertidumbre. El score ordena activos; no constituye una recomendación de compra o venta.